In [ ]:
import pandas as pd
import os

train = pd.read_csv("/content/drive/MyDrive/adult_train.csv")
test = pd.read_csv("/content/drive/MyDrive/adult_test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train columns:", train.columns.tolist())

df = pd.concat([train, test], ignore_index=True)

output_dir = "/content/drive/RA"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "adult_full.csv")
df.to_csv(output_path, index=False)

print(f"Combined dataset saved successfully at: {output_path}")
print("Shape of combined dataset:", df.shape)
print(df.head())


In [ ]:
import pandas as pd
import numpy as np
import os


src_path = "/content/drive/MyDrive/adult_full.csv"
save_dir = "/content/drive/MyDrive"
os.makedirs(save_dir, exist_ok=True)

df = pd.read_csv(src_path)

df = df.replace({'?': np.nan, ' ?': np.nan})
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip()


if "Target" in df.columns:
    df["Target"] = df["Target"].str.replace(".", "", regex=False).str.strip()


df.replace({'': np.nan}, inplace=True)

before = df.shape[0]
df.dropna(inplace=True)
after = df.shape[0]


clean_path = f"{save_dir}/adult_full_clean.csv"
df.to_csv(clean_path, index=False)


print("Cleaned file saved:", clean_path)
print(f"Rows before: {before} | after cleaning: {after}")
print("Columns:", list(df.columns))


if "Target" in df.columns:
    print("\nTarget value counts (normalized):")
    print(df["Target"].value_counts(normalize=True))
print(df.head())


In [ ]:
#Baseline ALM

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

clean_path = "/content/drive/MyDrive/adult_full_clean.csv"


df = pd.read_csv(clean_path)


train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["Target"]
)

out_dir = os.path.dirname(clean_path)
train_path = os.path.join(out_dir, "train.csv")
test_path  = os.path.join(out_dir, "test.csv")
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved splits:")
print("Train:", train_path, train_df.shape)
print("Test :", test_path,  test_df.shape)
print("\nTarget distribution (train):")
print(train_df["Target"].value_counts(normalize=True))
print("\nTarget distribution (test):")
print(test_df["Target"].value_counts(normalize=True))


In [ ]:
!pip -q install realtabformer

import pandas as pd
from pathlib import Path

from realtabformer import REaLTabFormer

In [ ]:
import torch, pandas as pd
from pathlib import Path
from realtabformer import REaLTabFormer

TRAIN_CSV = Path("./train.csv")
SAVE_DIR  = Path("./RTF_resources/rtf_model")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_CSV)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

rtf = REaLTabFormer(
    model_type="tabular",
    epochs=50,
    batch_size=128,
    learning_rate=2e-4,

    logging_strategy="steps",
    logging_steps=50,
    report_to=[],
)

rtf.fit(train_df, device=device)

try:
    rtf.save(SAVE_DIR)
    print(f"Model saved to: {SAVE_DIR}")
except Exception as e:
    print("Save not supported in this version:", e)

In [ ]:
from pathlib import Path
import torch, pandas as pd

OUT_DIR = Path("./RTF_resources/rtf_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Sampling on:", device)

SYNTH_ROWS = len(train_df)

try:
    synthetic_df = rtf.sample(n_samples=SYNTH_ROWS, gen_batch=128, device=device)
except TypeError:
    synthetic_df = rtf.sample(n=SYNTH_ROWS, gen_batch=128, device=device)

synthetic_path = OUT_DIR / "synthetic.csv"
synthetic_df.to_csv(synthetic_path, index=False)
print(f"Generated {len(synthetic_df)} rows → {synthetic_path}")


In [ ]:


import json, math
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import normalized_mutual_info_score, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from scipy.stats import ks_2samp, entropy, spearmanr


DATA_DIR = Path("./RTF_resources")
REAL_CSV = DATA_DIR / "train.csv"
SYN_CSV  = DATA_DIR / "rtf_outputs/synthetic.csv"
OUT_DIR  = DATA_DIR / "rtf_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

real = pd.read_csv(REAL_CSV)
synt = pd.read_csv(SYN_CSV)


target_col = "Target" if "Target" in real.columns else None
cols = [c for c in real.columns if c != target_col]
cat_cols = [c for c in cols if real[c].dtype == "object"]
num_cols = [c for c in cols if c not in cat_cols]


temp_cols = []

print("Columns:", cols)
print("Numeric:", num_cols)
print("Categorical:", cat_cols)
print("Temporal:", temp_cols)


def tvd_from_counts(p, q):

    keys = sorted(set(p).union(q), key=str)
    pv = np.array([p.get(k, 0.0) for k in keys], dtype=float)
    qv = np.array([q.get(k, 0.0) for k in keys], dtype=float)
    return 0.5 * np.abs(pv - qv).sum()

def js_divergence(p, q, base=np.e):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5*(p+q)
    return 0.5*entropy(p, m, base=base) + 0.5*entropy(q, m, base=base)

def hist_compare(xr, xs, bins=30, rng=None, smooth=1e-9):
    if rng is None:
        lo = np.nanmin(np.concatenate([xr, xs])); hi = np.nanmax(np.concatenate([xr, xs]))
        if lo == hi: hi = lo + 1.0
        rng = (lo, hi)
    hr, edges = np.histogram(xr, bins=bins, range=rng)
    hs, _     = np.histogram(xs, bins=bins, range=rng)
    pr = (hr + smooth) / (hr.sum() + smooth*bins)
    ps = (hs + smooth) / (hs.sum() + smooth*bins)
    return dict(kl_rs=float(entropy(pr, ps)),
                kl_sr=float(entropy(ps, pr)),
                jsd=float(js_divergence(pr, ps)))

def safe_mean(vals):
    arr = np.array([v for v in vals if pd.notna(v)], dtype=float)
    return float(arr.mean()) if arr.size else float("nan")

def inv01(x, cap=1.0):
    if pd.isna(x): return np.nan
    return float(max(0.0, min(1.0, 1.0 - x/cap)))


NAN_TOKEN = "___NaN___"
real_cat = real[cat_cols].copy().astype(str)
synt_cat = synt[cat_cols].copy().astype(str)

for df_ in (real_cat, synt_cat):
    for c in cat_cols:
        df_[c] = df_[c].replace({"nan": NAN_TOKEN}).str.strip()


D = {"numeric": {}, "categorical": {}}


for c in num_cols:
    xr = real[c].dropna().values
    xs = synt[c].dropna().values
    if len(xr)==0 or len(xs)==0:
        D["numeric"][c] = {"ks": np.nan, "kl_rs": np.nan, "kl_sr": np.nan, "jsd": np.nan, "range_coverage": np.nan}
        continue
    ks = ks_2samp(xr, xs).statistic
    h = hist_compare(xr, xs, bins=30)
    lo, hi = np.nanmin(xr), np.nanmax(xr)
    rc = float(np.mean((xs >= lo) & (xs <= hi)))
    D["numeric"][c] = {"ks": float(ks), **h, "range_coverage": rc}


for c in cat_cols:
    pr = real_cat[c].value_counts(normalize=True).to_dict()
    ps = synt_cat[c].value_counts(normalize=True).to_dict()
    tvd = tvd_from_counts(pr, ps)
    D["categorical"][c] = {"tvd": float(tvd), "freq_alignment": float(1.0 - tvd)}

D_summary = dict(
    mean_numeric_ks=safe_mean([D["numeric"][c]["ks"] for c in num_cols]),
    mean_numeric_jsd=safe_mean([D["numeric"][c]["jsd"] for c in num_cols]),
    mean_numeric_range_coverage=safe_mean([D["numeric"][c]["range_coverage"] for c in num_cols]),
    mean_categorical_tvd=safe_mean([D["categorical"][c]["tvd"] for c in cat_cols]),
)
print("\n=== D (Distribution) ==="); print(D_summary)


A = {"numeric_outlier_diff": {}, "categorical_rare_diff": {}}

for c in num_cols:
    xr = real[c].dropna().values
    xs = synt[c].dropna().values
    if len(xr)==0 or len(xs)==0:
        A["numeric_outlier_diff"][c] = np.nan
        continue
    q1, q3 = np.percentile(xr, [25, 75]); iqr = q3 - q1
    lo = q1 - 1.5*iqr; hi = q3 + 1.5*iqr
    rate_r = np.mean((xr < lo) | (xr > hi))
    rate_s = np.mean((xs < lo) | (xs > hi))
    A["numeric_outlier_diff"][c] = float(abs(rate_r - rate_s))


rare_thresh = 0.01
for c in cat_cols:
    pr = real_cat[c].value_counts(normalize=True)
    ps = synt_cat[c].value_counts(normalize=True)
    rare_cats = pr[pr <= rare_thresh].index
    diffs = [abs(pr.get(cat, 0.0) - ps.get(cat, 0.0)) for cat in rare_cats]
    A["categorical_rare_diff"][c] = float(np.mean(diffs)) if diffs else 0.0

A_summary = dict(
    mean_numeric_outlier_rate_absdiff=safe_mean(list(A["numeric_outlier_diff"].values())),
    mean_categorical_rare_freq_absdiff=safe_mean(list(A["categorical_rare_diff"].values()))
)
print("\n=== A (Anomalies & rare) ==="); print(A_summary)


def spearman_matrix(df, cols):
    m = len(cols); R = np.ones((m, m)) * np.nan
    for i, a in enumerate(cols):
        for j, b in enumerate(cols):
            if i == j:
                R[i, j] = 1.0
            elif i < j:
                rho, _ = spearmanr(df[a], df[b], nan_policy="omit")
                R[i, j] = R[j, i] = rho
    return R

C_num = np.nan
if len(num_cols) >= 2:
    Rr = spearman_matrix(real, num_cols)
    Rs = spearman_matrix(synt, num_cols)
    mask = np.isfinite(Rr) & np.isfinite(Rs)
    C_num = float(1.0 - np.nanmean(np.abs(Rr[mask] - Rs[mask])))

def nmi_matrix(df, cols):
    m = len(cols); M = np.ones((m, m)) * np.nan
    for i, a in enumerate(cols):
        for j, b in enumerate(cols):
            if i == j:
                M[i, j] = 1.0
            elif i < j:
                M[i, j] = M[j, i] = normalized_mutual_info_score(df[a].astype(str), df[b].astype(str))
    return M


C_cat = np.nan
if len(cat_cols) >= 2:
    Mr = nmi_matrix(real_cat, cat_cols)
    Ms = nmi_matrix(synt_cat, cat_cols)
    mask = np.isfinite(Mr) & np.isfinite(Ms)
    C_cat = float(1.0 - np.nanmean(np.abs(Mr[mask] - Ms[mask])))

def discretize_series(x, bins=10):
    try:    return pd.qcut(x, q=bins, duplicates="drop").astype(str)
    except: return pd.cut(x, bins=bins, include_lowest=True).astype(str)

C_mix_vals = []
for a in num_cols:
    xr = discretize_series(real[a]); xs = discretize_series(synt[a])
    for b in cat_cols:
        nr = normalized_mutual_info_score(xr.astype(str), real_cat[b].astype(str))
        ns = normalized_mutual_info_score(xs.astype(str), synt_cat[b].astype(str))
        if pd.notna(nr) and pd.notna(ns):
            C_mix_vals.append(1.0 - abs(nr - ns))
C_mix = float(np.mean(C_mix_vals)) if C_mix_vals else np.nan

C_summary = dict(
    association_numeric_numeric=C_num,
    association_categorical_categorical=C_cat,
    association_mixed=C_mix
)
print("\n=== C (Associations) ==="); print(C_summary)


V = {"category_coverage": {}, "support_coverage_proxy": {}}
alpha = 1e-4
for c in cat_cols:
    pr = real_cat[c].value_counts(normalize=True)
    ps = synt_cat[c].value_counts(normalize=True)
    cats = set(pr.index)
    cc = float(np.mean([cat in ps.index for cat in cats]))
    num = 0.0; den = 0.0
    for cat, pr_k in pr.items():
        den += pr_k
        num += min(ps.get(cat, 0.0) + alpha, pr_k)
    sc = float(num / (den + 1e-12))
    V["category_coverage"][c] = cc
    V["support_coverage_proxy"][c] = sc

V_summary = dict(
    mean_category_coverage=safe_mean(list(V["category_coverage"].values())),
    mean_support_coverage=safe_mean(list(V["support_coverage_proxy"].values()))
)
print("\n=== V (Diversity/Coverage) ==="); print(V_summary)





D_score = np.nanmean([
    1.0 - (D_summary["mean_numeric_ks"]),
    1.0 - (D_summary["mean_numeric_jsd"]),
    D_summary["mean_numeric_range_coverage"],
    1.0 - (D_summary["mean_categorical_tvd"]),
])

A_score = np.nanmean([
    1.0 - min(1.0, A_summary["mean_numeric_outlier_rate_absdiff"]/0.10),
    1.0 - min(1.0, A_summary["mean_categorical_rare_freq_absdiff"]/0.02),
])

C_score = np.nanmean([
    C_summary["association_numeric_numeric"],
    C_summary["association_categorical_categorical"],
    C_summary["association_mixed"],
])

V_score = np.nanmean([
    V_summary["mean_category_coverage"],
    V_summary["mean_support_coverage"],
])


T_score = np.nan


WQ = {"D": 0.40, "A": 0.10, "C": 0.40, "V": 0.10, "T": 0.00}
Q_score = (
    WQ["D"] * (0 if pd.isna(D_score) else D_score) +
    WQ["A"] * (0 if pd.isna(A_score) else A_score) +
    WQ["C"] * (0 if pd.isna(C_score) else C_score) +
    WQ["V"] * (0 if pd.isna(V_score) else V_score) +
    WQ["T"] * (0 if pd.isna(T_score) else T_score)
)

print("\n=== Q aggregate ===")
print(Q_score)


Q_out = {
    "D_summary": D_summary, "A_summary": A_summary,
    "C_summary": C_summary, "V_summary": V_summary,
    "T_score": T_score, "Q_aggregate": float(Q_score)
}
import json, os
os.makedirs(OUT_DIR, exist_ok=True)
with open(OUT_DIR/"Q_metrics_full.json", "w") as f:
    json.dump(Q_out, f, indent=2)
print("Saved →", OUT_DIR/"Q_metrics_full.json")


In [ ]:


import json, numpy as np, pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import NearestNeighbors


DATA_DIR = Path("./RTF_resources")
REAL_CSV = DATA_DIR / "train.csv"
SYN_CSV  = DATA_DIR / "rtf_outputs/synthetic.csv"
OUT_DIR  = DATA_DIR / "rtf_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

real = pd.read_csv(REAL_CSV)
synt = pd.read_csv(SYN_CSV)

target_col = "Target" if "Target" in real.columns else None
cols = [c for c in real.columns if c != target_col]
cat_cols = [c for c in cols if real[c].dtype == "object"]
num_cols = [c for c in cols if c not in cat_cols]


from sklearn.preprocessing import OneHotEncoder
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


enc = ColumnTransformer([
    ("num", "passthrough", [c for c in cols if c in num_cols]),
    ("cat", make_ohe(),    [c for c in cols if c in cat_cols]),
])

X_real = enc.fit_transform(real[cols])
X_fake = enc.transform(synt[cols])

nn = NearestNeighbors(n_neighbors=1).fit(X_real)
dists, _ = nn.kneighbors(X_fake)
dists = dists.ravel()
DCR_mean = float(np.mean(dists))
DCR_std  = float(np.std(dists))


quantiles = np.linspace(0.05, 0.95, 19)
qd_vals = []
for c in num_cols:
    xr = np.asarray(real[c].dropna().values)
    xs = np.asarray(synt[c].dropna().values)
    if xr.size > 0 and xs.size > 0:
        qr = np.quantile(xr, quantiles)
        qs = np.quantile(xs, quantiles)
        qd_vals.append(float(np.mean(np.abs(qr - qs))))
Q_delta = float(np.mean(qd_vals)) if qd_vals else float("nan")


def norm_df(df, cols):
    out = df[cols].copy()
    for c in cols:
        if np.issubdtype(out[c].dtype, np.number):
            out[c] = out[c].round(6)
        else:
            out[c] = out[c].astype(str).str.strip()
    return out

real_norm = norm_df(real, cols)
fake_norm = norm_df(synt, cols)

dup_within_synth = float(1.0 - len(fake_norm.drop_duplicates()) / max(1, len(fake_norm)))
set_real = set(map(tuple, real_norm.to_numpy()))
set_fake = set(map(tuple, fake_norm.to_numpy()))
overlap_between = float(len(set_real & set_fake) / max(1, len(fake_norm)))


def normalize01(x, lo=0.0, hi=1.0):
    if pd.isna(x): return np.nan
    if hi == lo: return 0.0
    v = (x - lo) / (hi - lo)
    return float(max(0.0, min(1.0, v)))

def inv01(x, cap=1.0):
    if pd.isna(x): return np.nan
    return float(max(0.0, min(1.0, 1.0 - x / cap)))


d95 = float(np.percentile(dists, 95)) if len(dists) else 1.0
DCR_score = normalize01(DCR_mean, lo=0.0, hi=d95 if d95 > 0 else 1.0)

Qd_cap = float(np.nanmedian(qd_vals)) * 4 if qd_vals else (abs(Q_delta) + 1e-6)
Qdelta_score = inv01(Q_delta, cap=Qd_cap if Qd_cap > 0 else 1.0)

I_combined = 0.5 * (dup_within_synth + overlap_between)
I_score = inv01(I_combined, cap=0.10)


WP = {"DCR": 0.5, "Qdelta": 0.3, "I": 0.2}
P_aggregate = (
    WP["DCR"] * (0 if pd.isna(DCR_score) else DCR_score)
    + WP["Qdelta"] * (0 if pd.isna(Qdelta_score) else Qdelta_score)
    + WP["I"] * (0 if pd.isna(I_score) else I_score)
)


P_out = {
    "parts": {
        "DCR_mean": DCR_mean,
        "DCR_std": DCR_std,
        "Q_delta": Q_delta,
        "I_dup_within_synth": dup_within_synth,
        "I_overlap_between": overlap_between,
    },
    "scores": {
        "DCR_score": DCR_score,
        "Qdelta_score": Qdelta_score,
        "I_score": I_score,
    },
    "P_aggregate": float(P_aggregate),
}

print("\n=== P (Privacy) aggregate ===")
print(json.dumps(P_out, indent=2))
with open(OUT_DIR / "P_metrics_full.json", "w") as f:
    json.dump(P_out, f, indent=2)
print("Saved →", OUT_DIR / "P_metrics_full.json")


In [ ]:
#Sampling-time optimization with ALM

In [ ]:
!pip install -q realtabformer transformers datasets scikit-learn scipy


In [ ]:
from pathlib import Path
import json, os

BASE = Path("./RTF_resources")
ALM_OUT = BASE / "alm_outputs"
ALM_MODEL = BASE / "alm_model"
ALM_OUT.mkdir(parents=True, exist_ok=True)
ALM_MODEL.mkdir(parents=True, exist_ok=True)

print("ALM outputs  →", ALM_OUT.resolve())
print("ALM model    →", ALM_MODEL.resolve())


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import NearestNeighbors
from scipy.stats import ks_2samp, entropy
from sklearn.metrics import normalized_mutual_info_score
from scipy.stats import spearmanr


def score_quality_Q(real_df: pd.DataFrame, syn_df: pd.DataFrame, target_col="Target"):
    cols = [c for c in real_df.columns if c != target_col]
    cat_cols = [c for c in cols if real_df[c].dtype == "object"]
    num_cols = [c for c in cols if c not in cat_cols]

    def js_divergence(p, q, base=np.e):
        p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
        p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
        m = 0.5*(p+q);
        return 0.5*entropy(p, m, base=base)+0.5*entropy(q, m, base=base)
    def hist_compare(xr, xs, bins=30, rng=None, smooth=1e-9):
        if rng is None:
            lo = np.nanmin(np.concatenate([xr, xs])); hi = np.nanmax(np.concatenate([xr, xs]));
            if lo==hi: hi = lo+1.0; rng=(lo,hi)
        hr, edges = np.histogram(xr, bins=bins, range=rng)
        hs, _     = np.histogram(xs, bins=bins, range=rng)
        pr = (hr + smooth) / (hr.sum() + smooth*bins)
        ps = (hs + smooth) / (hs.sum() + smooth*bins)
        return {"jsd": float(js_divergence(pr, ps))}
    def tvd_from_counts(p, q):
        keys = sorted(set(p).union(q), key=str)
        pv = np.array([p.get(k, 0.0) for k in keys], dtype=float)
        qv = np.array([q.get(k, 0.0) for k in keys], dtype=float)
        return 0.5 * np.abs(pv - qv).sum()

    D_parts = []
    for c in num_cols:
        xr = real_df[c].dropna().values; xs = syn_df[c].dropna().values
        if xr.size and xs.size:
            ks = ks_2samp(xr, xs).statistic
            jsd = hist_compare(xr, xs)["jsd"]
            lo, hi = np.nanmin(xr), np.nanmax(xr)
            rc = float(np.mean((xs>=lo)&(xs<=hi)))
            D_parts.append( (1-ks, 1-jsd, rc) )
    D_score = np.mean([np.mean(p) for p in D_parts]) if D_parts else np.nan

    real_cat = real_df[cat_cols].copy().astype(str)
    syn_cat  = syn_df[cat_cols].copy().astype(str)
    tvds=[]
    for c in cat_cols:
        pr = real_cat[c].value_counts(normalize=True).to_dict()
        ps = syn_cat[c].value_counts(normalize=True).to_dict()
        tvds.append(1.0 - tvd_from_counts(pr, ps))
    D_cat = float(np.mean(tvds)) if tvds else np.nan


    def spearman_matrix(df, cols):
        m=len(cols); R=np.ones((m,m))*np.nan
        for i,a in enumerate(cols):
            for j,b in enumerate(cols):
                if i==j: R[i,i]=1.0
                elif i<j:
                    rho,_=spearmanr(df[a], df[b], nan_policy="omit")
                    R[i,j]=R[j,i]=rho
        return R
    C_num=np.nan
    if len(num_cols)>=2:
        Rr=spearman_matrix(real_df,num_cols); Rs=spearman_matrix(syn_df,num_cols)
        mask = np.isfinite(Rr)&np.isfinite(Rs)
        C_num = float(1.0 - np.nanmean(np.abs(Rr[mask]-Rs[mask])))

    def nmi_matrix(df, cols):
        m=len(cols); M=np.ones((m,m))*np.nan
        for i,a in enumerate(cols):
            for j,b in enumerate(cols):
                if i==j: M[i,i]=1.0
                elif i<j: M[i,j]=M[j,i]=normalized_mutual_info_score(df[a].astype(str), df[b].astype(str))
        return M
    C_cat=np.nan
    if len(cat_cols)>=2:
        Mr=nmi_matrix(real_cat,cat_cols); Ms=nmi_matrix(syn_cat,cat_cols)
        mask=np.isfinite(Mr)&np.isfinite(Ms)
        C_cat = float(1.0 - np.nanmean(np.abs(Mr[mask]-Ms[mask])))


    CC=[]
    for c in cat_cols:
        pr=real_cat[c].value_counts(normalize=True)
        ps=syn_cat[c].value_counts(normalize=True)
        cats=set(pr.index)
        cc=float(np.mean([k in ps.index for k in cats]))
        CC.append(cc)
    V_score = float(np.mean(CC)) if CC else np.nan


    parts = []
    for x in [D_score, D_cat, C_num, C_cat, V_score]:
        if not np.isnan(x): parts.append(x)
    Q = float(np.mean(parts)) if parts else 0.0
    return Q


def score_privacy_P(real_df: pd.DataFrame, syn_df: pd.DataFrame, target_col="Target"):
    cols = [c for c in real_df.columns if c != target_col]
    cat_cols = [c for c in cols if real_df[c].dtype == "object"]
    num_cols = [c for c in cols if c not in cat_cols]

    def make_ohe():
        try: return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        except TypeError: return OneHotEncoder(handle_unknown="ignore", sparse=False)

    enc = ColumnTransformer([
        ("num", "passthrough", num_cols),
        ("cat", make_ohe(),    cat_cols),
    ])
    X_real = enc.fit_transform(real_df[cols]); X_fake = enc.transform(syn_df[cols])


    nn = NearestNeighbors(n_neighbors=1).fit(X_real)
    dists,_ = nn.kneighbors(X_fake); dists=dists.ravel()
    DCR_mean = float(np.mean(dists))


    quantiles = np.linspace(0.05,0.95,19); qd_vals=[]
    for c in num_cols:
        xr = np.asarray(real_df[c].dropna().values)
        xs = np.asarray(syn_df[c].dropna().values)
        if xr.size and xs.size:
            qr = np.quantile(xr, quantiles); qs = np.quantile(xs, quantiles)
            qd_vals.append(float(np.mean(np.abs(qr-qs))))
    Q_delta = float(np.mean(qd_vals)) if qd_vals else float("nan")


    def norm_df(df, cols):
        out=df[cols].copy()
        for c in cols:
            if np.issubdtype(out[c].dtype, np.number): out[c]=out[c].round(6)
            else: out[c]=out[c].astype(str).str.strip()
        return out
    real_norm = norm_df(real_df, cols); fake_norm = norm_df(syn_df, cols)
    dup_within_synth = float(1.0 - len(fake_norm.drop_duplicates())/max(1,len(fake_norm)))
    set_real=set(map(tuple, real_norm.to_numpy())); set_fake=set(map(tuple, fake_norm.to_numpy()))
    overlap_between = float(len(set_real & set_fake)/max(1,len(fake_norm)))


    def normalize01(x, lo=0.0, hi=1.0):
        if np.isnan(x): return np.nan
        if hi==lo: return 0.0
        v=(x-lo)/(hi-lo); return float(max(0.0, min(1.0, v)))
    def inv01(x, cap=1.0):
        if np.isnan(x): return np.nan
        return float(max(0.0, min(1.0, 1.0 - x/cap)))

    d95 = float(np.percentile(dists, 95)) if len(dists) else 1.0
    DCR_score = normalize01(DCR_mean, lo=0.0, hi=d95 if d95>0 else 1.0)
    Qd_cap = float(np.nanmedian(qd_vals))*4 if qd_vals else (abs(Q_delta)+1e-6)
    Qdelta_score = inv01(Q_delta, cap=Qd_cap if Qd_cap>0 else 1.0)
    I_combined = 0.5*(dup_within_synth + overlap_between)
    I_score = inv01(I_combined, cap=0.10)

    P = 0.5*DCR_score + 0.3*Qdelta_score + 0.2*I_score
    return float(P)



In [ ]:
from pathlib import Path
import json, pandas as pd
import numpy as np


BASE      = Path("./")
ALM_OUT   = BASE / "rtf_alm_outputs"
ALM_MODEL = BASE / "rtf_alm_model"
ALM_OUT.mkdir(parents=True, exist_ok=True)
ALM_MODEL.mkdir(parents=True, exist_ok=True)


T_best     = best["T"]
top_p_best = best["top_p"]

print(f"Using best ALM settings: T={T_best}, top_p={top_p_best}")


SYNTH_ROWS = len(REAL)

try:
    synthetic_df = rtf.sample(
        n_samples=SYNTH_ROWS,
        temperature=T_best,
        top_p=top_p_best
    )
except TypeError:
    synthetic_df = rtf.sample(
        n=SYNTH_ROWS,
        temperature=T_best,
        top_p=top_p_best
    )


final_csv = ALM_OUT / "synthetic_alm.csv"
synthetic_df.to_csv(final_csv, index=False)


Q_final = score_quality_Q(REAL, synthetic_df)
P_final = score_privacy_P(REAL, synthetic_df)

metrics = {
    "Q_final": float(Q_final),
    "P_final": float(P_final),
    "Pmin": float(Pmin),
    "lambda": float(lam),
    "mu": float(mu),
    "temperature": float(T_best),
    "top_p": float(top_p_best),
}
with open(ALM_OUT / "alm_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved synthetic  →", final_csv)
print("Saved metrics    →", ALM_OUT / "alm_metrics.json")
print(f"Final: Q={Q_final:.3f}, P={P_final:.3f} at T={T_best}, top_p={top_p_best}")


conf = {
    "checkpoint_dir": "./RTF_resources/rtf_model",
    "sampling": {
        "temperature": float(T_best),
        "top_p": float(top_p_best)
    },
    "alm": {
        "Pmin": float(Pmin),
        "lambda": float(lam),
        "mu": float(mu)
    }
}
with open(ALM_MODEL / "config.json", "w") as f:
    json.dump(conf, f, indent=2)

print("Saved ALM config →", (ALM_MODEL / "config.json").resolve())


In [ ]:
import os, glob, json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


state_files = []
state_files += glob.glob("./rtf_model/**/trainer_state.json", recursive=True)
state_files += glob.glob("./RTF_resources/rtf_model/**/trainer_state.json", recursive=True)

if not state_files:
    print(" No trainer_state.json found under ./rtf_model or ./RTF_resources/rtf_model")
    print("You need to re-train with rtf.fit(...) in this environment so the trainer can save logs.")
else:

    state_files = sorted(state_files, key=os.path.getmtime)
    state_path = state_files[-1]
    print("Using trainer state:", state_path)

    with open(state_path, "r") as f:
        state = json.load(f)

    log_hist = state.get("log_history", [])
    if not log_hist:
        print(" trainer_state.json has no log_history; nothing to plot.")
    else:

        epoch_losses = defaultdict(list)
        for rec in log_hist:
            if "loss" in rec and "epoch" in rec:
                epoch_losses[rec["epoch"]].append(rec["loss"])

        epochs = sorted(epoch_losses.keys())
        mean_losses = [np.mean(epoch_losses[e]) for e in epochs]

        plt.figure(figsize=(6, 4))
        plt.plot(epochs, mean_losses, marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Training loss")
        plt.title("REaLTabFormer training loss per epoch")
        plt.grid(True)
        plt.tight_layout()
        plt.show()


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


if "rtf" not in globals() or rtf is None:
    raise RuntimeError(
        "`rtf` is not defined.\n"
        "Run your training cell first, e.g.:\n"
        "  rtf = REaLTabFormer(...)\n"
        "  rtf.fit(train_df, device=...)\n"
    )
else:
    print("Using existing `rtf` from memory.")


REAL = pd.read_csv("./train.csv")

Pmin = 0.80
lam  = 0.0
mu   = 1.0
alpha_mu = 2.0
patience = 3

temps = [0.7, 0.9, 1.0, 1.1, 1.3]
topps = [0.90, 0.95, 0.98, 0.99, 1.0]

best = None
bad_rounds = 0
alm_losses = []

print("\nStarting ALM search...")
for outer in range(3):
    results = []
    for T in temps:
        for Pcut in topps:
            n_synth = int(min(len(REAL), 10000))

            try:
                syn_df = rtf.sample(n_samples=n_synth, temperature=T, top_p=Pcut)
            except TypeError:
                syn_df = rtf.sample(n=n_synth, temperature=T, top_p=Pcut)

            Q = score_quality_Q(REAL, syn_df)
            P = score_privacy_P(REAL, syn_df)

            gap = Pmin - P
            penalty = max(0.0, gap)
            L = (1.0 - Q) + lam * penalty + 0.5 * mu * (penalty ** 2)

            results.append({"T": T, "top_p": Pcut, "Q": Q, "P": P, "L": L})


    results = sorted(results, key=lambda r: r["L"])
    cand = results[0]
    alm_losses.append(cand["L"])

    print(
        f"[outer {outer}] best so far: "
        f"T={cand['T']}, top_p={cand['top_p']}, "
        f"Q={cand['Q']:.3f}, P={cand['P']:.3f}, L={cand['L']:.4f}"
    )

    if cand["P"] < Pmin:
        lam = lam + (Pmin - cand["P"])
        bad_rounds += 1
        if bad_rounds >= patience:
            mu *= alpha_mu
            bad_rounds = 0
    else:
        lam = max(0.0, lam * 0.5)
        bad_rounds = 0

    best = cand

print("\nALM search finished. Best config:")
print(best)
print("ALM losses per outer iteration:", alm_losses)


iters = list(range(len(alm_losses)))
plt.figure(figsize=(6, 4))
plt.plot(iters, alm_losses, marker="o")
plt.xlabel("ALM outer iteration")
plt.ylabel("ALM loss L")
plt.title("ALM objective over outer iterations")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


if "rtf" not in globals() or rtf is None:
    raise RuntimeError("`rtf` is not defined. Run your training cell first.")


REAL = pd.read_csv("./train.csv")


outer_iters = 3
temps       = [0.9, 1.1]
topps       = [0.90, 0.99]

Pmin     = 0.80
lam      = 0.0
mu       = 1.0
alpha_mu = 2.0
patience = 3

best = None
bad_rounds = 0
alm_losses = []

print("\nStarting FAST ALM search (for plotting)...")
for outer in range(outer_iters):
    results = []
    for T in temps:
        for Pcut in topps:

            n_synth = int(min(len(REAL), 3000))

            try:
                syn_df = rtf.sample(n_samples=n_synth, temperature=T, top_p=Pcut)
            except TypeError:
                syn_df = rtf.sample(n=n_synth, temperature=T, top_p=Pcut)

            Q = score_quality_Q(REAL, syn_df)
            P = score_privacy_P(REAL, syn_df)

            gap = Pmin - P
            penalty = max(0.0, gap)
            L = (1.0 - Q) + lam * penalty + 0.5 * mu * (penalty ** 2)

            results.append({"T": T, "top_p": Pcut, "Q": Q, "P": P, "L": L})


    results = sorted(results, key=lambda r: r["L"])
    cand = results[0]
    alm_losses.append(cand["L"])

    print(
        f"[outer {outer}] best so far: "
        f"T={cand['T']}, top_p={cand['top_p']}, "
        f"Q={cand['Q']:.3f}, P={cand['P']:.3f}, L={cand['L']:.4f}"
    )


    if cand["P"] < Pmin:
        lam = lam + (Pmin - cand["P"])
        bad_rounds += 1
        if bad_rounds >= patience:
            mu *= alpha_mu
            bad_rounds = 0
    else:
        lam = max(0.0, lam * 0.5)
        bad_rounds = 0

    best = cand

print("\nFAST ALM search finished. Best config:")
print(best)
print("ALM losses per outer iteration:", alm_losses)


iters = list(range(len(alm_losses)))
plt.figure(figsize=(6, 4))
plt.plot(iters, alm_losses, marker="o")
plt.xlabel("ALM outer iteration")
plt.ylabel("ALM loss L")
plt.title("ALM objective over outer iterations")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
#Training-time optimization with ALM

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch


DATA_PATH = Path("./train.csv")
REAL_FULL = pd.read_csv(DATA_PATH)

print("Full real data shape:", REAL_FULL.shape)
print("Columns:", REAL_FULL.columns.tolist())

train_inner_df, val_df = train_test_split(
    REAL_FULL,
    test_size=0.2,
    random_state=42,
    stratify=REAL_FULL["Target"]
)

print("Inner-train shape:", train_inner_df.shape)
print("Validation shape :", val_df.shape)


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:
from realtabformer import REaLTabFormer




def train_and_eval_rtf(hparams, train_df, val_df, device="cpu", n_synth_val=2000):
    
    print(f"\nTraining trial with hparams: {hparams}")


    rtf = REaLTabFormer(
        model_type="tabular",
        epochs=hparams["epochs"],
        batch_size=hparams["batch_size"],
        learning_rate=hparams["learning_rate"],
        logging_strategy="steps",
        logging_steps=50,
        report_to=[],
    )


    rtf.fit(train_df, device=device)


    try:
        syn_df = rtf.sample(n_samples=n_synth_val)
    except TypeError:
        syn_df = rtf.sample(n=n_synth_val)


    Q_val = score_quality_Q(val_df, syn_df, target_col="Target")
    print(f"Q_val for this trial = {Q_val:.4f}")

    return Q_val, rtf


In [ ]:
!pip install -q optuna

import optuna

def objective(trial: optuna.Trial):

    lr = trial.suggest_loguniform("learning_rate", 1e-5, 5e-4)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    epochs = trial.suggest_int("epochs", 20, 50)

    hparams = {
        "learning_rate": lr,
        "batch_size": batch_size,
        "epochs": epochs,
    }


    Q_val, _ = train_and_eval_rtf(
        hparams=hparams,
        train_df=train_inner_df,
        val_df=val_df,
        device=device,
        n_synth_val=2000,
    )

    return Q_val


study = optuna.create_study(direction="maximize")


study.optimize(objective, n_trials=5)

print("\n=== Best trial from Bayesian Optimization ===")
print("  Best Q_val      :", study.best_value)
print("  Best parameters :", study.best_params)


In [ ]:
import random

LR_CHOICES = [1e-5, 5e-5, 1e-4, 2e-4, 5e-4]
BATCH_CHOICES = [64, 128, 256]

N_TRIALS_STAGE1 = 4
TOP_K = 2

results_stage1 = []

print("\n=== Bandit Stage 1: quick screening with fewer epochs ===")
for i in range(N_TRIALS_STAGE1):

    lr = random.choice(LR_CHOICES)
    batch_size = random.choice(BATCH_CHOICES)
    epochs = 8

    hparams = {
        "learning_rate": lr,
        "batch_size": batch_size,
        "epochs": epochs,
    }

    print(f"\n[Stage 1 - Trial {i+1}/{N_TRIALS_STAGE1}]  hparams={hparams}")
    Q_val, _ = train_and_eval_rtf(
        hparams=hparams,
        train_df=train_inner_df,
        val_df=val_df,
        device=device,
        n_synth_val=800,
    )

    results_stage1.append({
        "hparams": hparams,
        "Q_val": Q_val,
    })


results_stage1_sorted = sorted(results_stage1, key=lambda x: x["Q_val"], reverse=True)
top_candidates = results_stage1_sorted[:TOP_K]

print("\nTop configs from Stage 1:")
for c in top_candidates:
    print("  Q_val={:.4f}  hparams={}".format(c["Q_val"], c["hparams"]))


print("\n=== Bandit Stage 2: full training on top candidates ===")
best_bandit_Q = -1e9
best_bandit_hparams = None
best_bandit_model = None

for idx, cand in enumerate(top_candidates):

    full_hparams = cand["hparams"].copy()
    full_hparams["epochs"] = 25

    print(f"\n[Stage 2 - Candidate {idx+1}/{TOP_K}]  full_hparams={full_hparams}")
    Q_val_full, rtf_model = train_and_eval_rtf(
        hparams=full_hparams,
        train_df=train_inner_df,
        val_df=val_df,
        device=device,
        n_synth_val=1500,
    )

    if Q_val_full > best_bandit_Q:
        best_bandit_Q = Q_val_full
        best_bandit_hparams = full_hparams
        best_bandit_model = rtf_model

print("\n=== Best result from Bandit Search ===")
print("  Best Q_val      :", best_bandit_Q)
print("  Best parameters :", best_bandit_hparams)


In [ ]:
print("\n=== COMPARING BAYESIAN vs BANDIT ===")
print(f"Best Bayesian Q_val = {study.best_value}")
print(f"Best Bandit  Q_val = {best_bandit_Q}")


if study.best_value >= best_bandit_Q:
    print("\nBayesian Optimization wins! Using Bayesian hyperparameters.")
    final_hparams = {
        "learning_rate": study.best_params["learning_rate"],
        "batch_size": study.best_params["batch_size"],
        "epochs": study.best_params["epochs"],
    }
else:
    print("\nBandit Search wins! Using Bandit hyperparameters.")
    final_hparams = best_bandit_hparams

print("\nChosen hyperparameters BEFORE capping epochs:")
print(final_hparams)


MAX_FINAL_EPOCHS = 25
final_hparams["epochs"] = min(final_hparams["epochs"], MAX_FINAL_EPOCHS)

print(f"\nUsing epochs={final_hparams['epochs']} for the final model (capped).")


from realtabformer import REaLTabFormer

print("\n=== TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS (CAPPED) ===")

final_rtf = REaLTabFormer(
    model_type="tabular",
    epochs=final_hparams["epochs"],
    batch_size=final_hparams["batch_size"],
    learning_rate=final_hparams["learning_rate"],
    logging_strategy="steps",
    logging_steps=50,
    report_to=[],
)

final_rtf.fit(REAL_FULL, device=device)
print("Final model trained with tuned (and capped) hyperparameters.")


rtf = final_rtf
print("`rtf` now points to the tuned final model. You can run your ALM search cell next.")


In [ ]:

from pathlib import Path
import json, numpy as np, pandas as pd


if "rtf" not in globals() or rtf is None:
    raise RuntimeError(
        "`rtf` (final model) is not defined.\n"
        "Run Cell 5 first to train the final tuned REaLTabFormer model."
    )
else:
    print("Using final tuned `rtf` model for ALM search.")


ALM_OUT   = Path(".") / "alm_outputs"
ALM_MODEL = Path(".") / "alm_model"
ALM_OUT.mkdir(parents=True, exist_ok=True)
ALM_MODEL.mkdir(parents=True, exist_ok=True)

print("Local ALM outputs dir →", ALM_OUT.resolve())
print("Local ALM model dir   →", ALM_MODEL.resolve())


REAL = REAL_FULL
print("REAL shape for ALM:", REAL.shape)


Pmin = 0.80
lam  = 0.0
mu   = 1.0
alpha_mu = 2.0
patience = 3


temps = [0.7, 0.9, 1.0, 1.1, 1.3]
topps = [0.90, 0.95, 0.98, 0.99, 1.0]

best = None
bad_rounds = 0
history = []

print("\n=== Starting ALM search over (T, top_p) ===")
for outer in range(3):
    results = []
    print(f"\n[Outer iteration {outer}] ----------------------------")
    for T in temps:
        for Pcut in topps:

            n_synth = int(min(len(REAL), 10000))


            try:
                syn_df = rtf.sample(
                    n_samples=n_synth,
                    temperature=T,
                    top_p=Pcut
                )
            except TypeError:
                syn_df = rtf.sample(
                    n=n_synth,
                    temperature=T,
                    top_p=Pcut
                )


            Q = score_quality_Q(REAL, syn_df, target_col="Target")
            P = score_privacy_P(REAL, syn_df, target_col="Target")


            gap = Pmin - P
            penalty = max(0.0, gap)
            L = (1.0 - Q) + lam * penalty + 0.5 * mu * (penalty ** 2)

            res = {
                "T": T,
                "top_p": Pcut,
                "Q": Q,
                "P": P,
                "L": L,
                "lam": lam,
                "mu": mu,
                "outer": outer
            }
            results.append(res)
            history.append(res)

            print(
                f"  T={T:.2f}, top_p={Pcut:.2f} → "
                f"Q={Q:.3f}, P={P:.3f}, L={L:.4f}"
            )


    results = sorted(results, key=lambda r: r["L"])
    cand = results[0]

    print(
        f"\n[Outer {outer}] best candidate by L: "
        f"T={cand['T']}, top_p={cand['top_p']}, "
        f"Q={cand['Q']:.3f}, P={cand['P']:.3f}, L={cand['L']:.4f}"
    )

    if cand["P"] < Pmin:

        lam = lam + (Pmin - cand["P"])
        bad_rounds += 1
        if bad_rounds >= patience:
            mu *= alpha_mu
            bad_rounds = 0
    else:

        lam = max(0.0, lam * 0.5)
        bad_rounds = 0

    best = cand

print("\n=== ALM search finished. Best (T, top_p) by ALM loss (L): ===")
print(best)


best_by_privacy = max(history, key=lambda r: r["P"])
print("\n=== Config with highest privacy P in ALM history ===")
print(best_by_privacy)


BASE_SAVE = Path("/home/RA/RTF_resources")
BASE_SAVE.mkdir(parents=True, exist_ok=True)

path_history = BASE_SAVE / "alm_search_history.json"
path_conf    = BASE_SAVE / "alm_best_config.json"

with open(path_history, "w") as f:
    json.dump(history, f, indent=2)

config_to_save = {
    "best_tradeoff_by_L": best,
    "best_by_privacy_P": best_by_privacy,
}
with open(path_conf, "w") as f:
    json.dump(config_to_save, f, indent=2)

print("\nSaved full ALM search history →", path_history.resolve())
print("Saved best ALM config        →", path_conf.resolve())


In [ ]:


from pathlib import Path
import json, pandas as pd

BASE_SAVE = Path("/home/RA/RTF_resources")
BASE_SAVE.mkdir(parents=True, exist_ok=True)

if "best_by_privacy" not in globals():
    raise RuntimeError(
        "`best_by_privacy` not found. Run Cell 6 (ALM search) before Cell 7."
    )


cfg = best_by_privacy


T_best     = cfg["T"]
top_p_best = cfg["top_p"]

print(f"\nUsing ALM settings for final synthetic data: T={T_best}, top_p={top_p_best}")


REAL = REAL_FULL
SYNTH_ROWS = len(REAL)


try:
    synthetic_df = rtf.sample(
        n_samples=SYNTH_ROWS,
        temperature=T_best,
        top_p=top_p_best
    )
except TypeError:
    synthetic_df = rtf.sample(
        n=SYNTH_ROWS,
        temperature=T_best,
        top_p=top_p_best
    )


final_csv = BASE_SAVE / "synthetic_alm.csv"
synthetic_df.to_csv(final_csv, index=False)


Q_final = score_quality_Q(REAL, synthetic_df, target_col="Target")
P_final = score_privacy_P(REAL, synthetic_df, target_col="Target")


metrics = {
    "Q_final": float(Q_final),
    "P_final": float(P_final),
    "Pmin": float(Pmin),
    "lambda": float(lam),
    "mu": float(mu),
    "temperature": float(T_best),
    "top_p": float(top_p_best),
    "used_config": "best_by_privacy_P",
}

metrics_path = BASE_SAVE / "alm_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("\nSaved ALM synthetic data  →", final_csv.resolve())
print("Saved ALM metrics         →", metrics_path.resolve())
print(f"Final (ALM): Q={Q_final:.3f}, P={P_final:.3f} at T={T_best}, top_p={top_p_best}")
